# YOLOv8n + DINOv2 on Pascal VOC (Kaggle runner)

**Before running** (right-hand *Settings* panel):
1. *Accelerator*: **GPU T4 x2** or **GPU P100** (only one GPU is used).
2. *Internet*: **On** (downloads Pascal VOC, YOLOv8n and DINOv2 weights).
3. *Add Input* → *Your Datasets* → the dataset holding `dino_yolo.py`, `train.py`, `evaluate.py`, `dino_analysis.py`, `requirements.txt`.

Run once with `QUICK_TEST = True` interactively to check everything, then set it to `False` and use **Save Version → Save & Run All (Commit)**.

In [ ]:
QUICK_TEST = False  # True: 1 epoch on 5% of the training data, to check the pipeline end to end
VARIANTS = ["baseline", "dino-cls", "dino-patch"]  # drop "dino-patch" to save ~1/3 of the time
EPOCHS = 20
PREV = ""  # to continue an interrupted version: attach its output as input, e.g. "/kaggle/input/<notebook-name>/runs"

In [ ]:
import glob, os, shutil

src = glob.glob("/kaggle/input/**/dino_yolo.py", recursive=True)
assert src, "Attach the dataset containing dino_yolo.py (Add Input -> Your Datasets)"
code_dir = os.path.dirname(src[0])
os.chdir("/kaggle/working")
for f in ["dino_yolo.py", "train.py", "evaluate.py", "dino_analysis.py", "requirements.txt"]:
    shutil.copy(os.path.join(code_dir, f), f)
if PREV:
    shutil.copytree(PREV, "runs", dirs_exist_ok=True)
print(code_dir, sorted(os.listdir()))

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, ultralytics, transformers; print(torch.__version__, torch.cuda.is_available(), ultralytics.__version__, transformers.__version__)"

In [ ]:
# Dataset lives in /tmp (not saved as notebook output); runs and results are saved in /kaggle/working.
!yolo settings datasets_dir=/tmp/datasets runs_dir=/kaggle/working/runs weights_dir=/kaggle/working/weights > /dev/null
!python -c "from ultralytics.data.utils import check_det_dataset; d = check_det_dataset('VOC.yaml'); print(d['nc'], 'classes')" 2>&1 | tail -3
!rm -f /tmp/datasets/VOC/images/*.zip && du -sh /tmp/datasets/VOC && df -h /tmp | tail -1

In [ ]:
import torch
import dino_yolo  # noqa: F401  (needed to unpickle YOLODinoModel checkpoints)


def status(v):
    last = f"runs/detect/{v}/weights/last.pt"
    if not os.path.exists(last):
        return "new"
    # Ultralytics sets epoch=-1 in last.pt once a run has finished
    return "done" if torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1 else "resume"


if QUICK_TEST:
    shutil.rmtree("runs", ignore_errors=True)
    os.makedirs("runs")
    open("runs/.quick", "w").close()
elif os.path.exists("runs/.quick"):
    shutil.rmtree("runs")  # discard quick-test runs before the real training

args = "--device 0 --cache ram --workers 4 " + ("--epochs 1 --fraction 0.05" if QUICK_TEST else f"--epochs {EPOCHS}")
for v in VARIANTS:
    s = status(v)
    print(f"===== {v}: {s} =====", flush=True)
    if s == "new":
        !python train.py --variant {v} {args}
    elif s == "resume":
        !python train.py --variant {v} --resume --device 0 --workers 4

In [ ]:
RUN_NAMES = " ".join(VARIANTS)
!python evaluate.py --runs {RUN_NAMES} --device 0
!python dino_analysis.py --device 0

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("results/metrics.md").read()))
display(Markdown(open("results/dino_knn_probe.md").read()))
for f in ["training_curves.png", "per_class_ap50.png", "dino_patch_pca.png"]:
    display(Image(f"results/{f}"))

# One file to download from the Output tab (last.pt checkpoints are excluded to keep it small)
!zip -qr results_bundle.zip results runs -x "runs/detect/*/weights/last.pt" && ls -lh results_bundle.zip